In [ ]:
import os
from PIL import Image
import torch
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler

In [ ]:
INPUT_IMG_PATH = './images/input/roland.png'
OUTPUT_DIR_PATH = './images/output/'
MODEL_ID = "timbrooks/instruct-pix2pix"

In [ ]:
def resize_image_keep_aspect(image, target_size=256):
    w, h = image.size

    if w == target_size or h == target_size: # do not resize if one dimension is already the target size
        print(f"Image already has one dimension of {target_size}, skipping resize.")
        return image
    
    if w > h:
        new_w = target_size
        new_h = int(h * (target_size / w))
    else:
        new_h = target_size
        new_w = int(w * (target_size / h))
    return image.resize((new_w, new_h))

def save_image(image, path):
    if os.path.exists(path):
        print(f"Image already exists at {path}, skipping save.")
        return
    with open(path, 'wb') as f:
        image.save(f, format='PNG')

In [ ]:
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, safety_checker=None)
pipe.to("cuda")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

# Image Manipulation

In [ ]:
# Loading and resizing the image
image = Image.open(INPUT_IMG_PATH).convert("RGB")
image_resized = resize_image_keep_aspect(image)
save_image(image=image_resized, path=INPUT_IMG_PATH)

In [ ]:
prompt = "make the building look like a castle"

In [ ]:
images = pipe(prompt, image=image_resized, num_inference_steps=10, image_guidance_scale=1).images
images[0]

In [ ]:
save_image(image=images[0], path=os.path.join(OUTPUT_DIR_PATH, 'roland_snowman.png'))